In [ ]:

import glob, bisect
import zipfile
import numpy as np
import zarr, torch
import os, random
import torch.nn.functional as Fa
from tqdm import tqdm
from torch.utils.data import Dataset, IterableDataset, DataLoader, Subset
from datasets import load_dataset
from zarr.storage import DirectoryStore
from functools import lru_cache
from typing import Optional
#from utils import *

In [3]:
# -------------------------
# Hyper params
# -------------------------
K = 1024
batch_size = 256 #4096
train_fraction=0.8
seed=42
model_variant = "base"
lr = 3e-4
max_steps=20000
eval_every=100
val_max_batches=200

MICRO_BATCH = 32 #64          # pick what fits (try 256, then 128 if still OOM)
GRAD_ACCUM_STEPS = batch_size // MICRO_BATCH     # 256*16 = 4096 effective batch

USE_AMP = True
AMP_DTYPE = torch.bfloat16  # or torch.float16

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



def train_fn():

    # reproducibility
    torch.manual_seed(int(seed))
    np.random.seed(int(seed))

    zarr_dirs = unzip_zarr_zips("../../data/order_model/train", "*_features.zarr.zip")  # creates train/*_features.zarr/
    ds = MultiDirZarrOrderDataset(zarr_dirs, seq_len=1024)
    
    train_dl, val_dl = make_train_val_loaders(
        ds,
        val_frac=0.01,
        seed=seed,
        batch_size= MICRO_BATCH,
        num_workers=10, #os.cpu_count() - 1,
        pin_memory=True,
        drop_last=True,
    )

    # -------------------------
    # Model selection (model_variant)
    # -------------------------
    model, model_hps = build_model_from_variant(str(model_variant))
    
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=float(lr))
    
    # -------------------------
    # Train loop
    # -------------------------
    model.train()
    train_it = iter(train_dl)
    
    pbar = tqdm(range(1, max_steps + 1), desc=f"train ({model_variant}, frac={train_fraction})")
    
    for step in pbar:
        opt.zero_grad(set_to_none=True)
    
        loss_accum = 0.0
    
        for _ in range(GRAD_ACCUM_STEPS):
            X = next(train_it)
            X = X.to(device, non_blocking=True)
    
            with torch.amp.autocast("cuda", enabled=(USE_AMP and device.type == "cuda"), dtype=AMP_DTYPE):
                logits = model(X)
                loss = lm_loss_all_positions(logits, X) / GRAD_ACCUM_STEPS  # scale loss
    
            loss.backward()
            loss_accum += loss.item()
    
        opt.step()
    
        pbar.set_postfix(train_loss=f"{loss_accum:.4f}", params=f"{n_params/1e6:.2f}M")
    
        if step % eval_every == 0:
            val_loss = compute_val_loss(model, val_dl, val_max_batches)
            print(f"\nstep {step:6d} | val_loss {val_loss:.4f} | params {n_params/1e6:.2f}M\n")
            model.train()

if __name__ == "__main__":
    train_fn()


train (base, frac=0.8):   0%|          | 99/20000 [03:56<13:11:15,  2.39s/it, params=10.18M, train_loss=7.4829] 


NameError: name 'compute_val_loss' is not defined